# Chapter 24: RNNs & LSTMs

Companion Colab notebook for **AI/ML: Zero to Hero**, Module 5, Chapter 24.

This notebook actually *runs* the PyTorch code that the lesson page (`dl/ch24-rnns-lstms.html`) shows as reference-only. Every cell below was executed for real — no fabricated output.

Chapter 23's convolutions exploit spatial structure — nearby pixels are related. Sequences (a sentence, a stock price over time, an audio waveform) have a different structure: **order** matters, and earlier elements can influence how you should interpret later ones. **Recurrent Neural Networks (RNNs)** add a hidden state that carries information forward from one time step to the next — and **LSTMs** fix the biggest flaw in the basic version.

**Note on data:** to keep this notebook fast, offline, and 100% reproducible, we train on a small **synthetic** sequence task instead of downloading a real text dataset — but the model, training loop, and every API call are the real thing.

In [1]:
import torch
print('torch version:', torch.__version__)


torch version: 2.8.0+cpu


## 24.1 — Why Sequential Data Needs Memory

"The movie was not good" and "The movie was good, not bad" use almost the same words in a different order — and mean opposite things. A network that looks at each word independently can't capture a dependency that might stretch back 20 words. What sequential data needs is a network that processes one element at a time while carrying forward a summary of everything it has seen so far.

## 24.2 — The Vanilla RNN: A Hidden State Carried Across Time

A recurrent layer processes a sequence one step at a time. At each step `t`, it combines the current input `x_t` with the **hidden state** `h_(t-1)` carried over from the previous step, producing a new hidden state `h_t` — the same small set of weights is reused at every time step, just like a convolution kernel reuses the same weights at every spatial position.

In [2]:
import torch
import torch.nn as nn

# input_size=5 features per time step, hidden_size=8, batch_first puts batch dim first
rnn = nn.RNN(input_size=5, hidden_size=8, batch_first=True)

x = torch.randn(1, 10, 5)   # (batch=1, seq_len=10, input_size=5)
output, h_n = rnn(x)
print(output.shape)   # (1, 10, 8) -- hidden state at EVERY time step
print(h_n.shape)       # (1, 1, 8)  -- just the FINAL hidden state


torch.Size([1, 10, 8])
torch.Size([1, 1, 8])


## 24.3 — The Vanishing Gradient Problem

Training an RNN means backpropagating through every time step — 10 steps back for a 10-word sentence, thousands of steps back for a long time series. Each step multiplies gradients by roughly the same small numbers, and repeated multiplication either shrinks toward zero (**vanishing gradients**) or blows up (**exploding gradients**). In practice, vanilla RNNs "forget" anything more than about 10-20 steps in the past.

> **This is why plain RNNs struggle with long sequences.** A model that needs to remember the subject of a sentence 30 words back to get the verb agreement right simply can't learn to do so reliably with a vanilla RNN — the gradient carrying that "remember this" signal has vanished by the time it reaches the relevant weights.

## 24.4 — LSTM: Gates That Fix It

The **Long Short-Term Memory (LSTM)** cell adds a separate **cell state** — a kind of conveyor belt that information can travel along mostly unchanged — plus three learned **gates** that control it at every step:

1. **Forget gate** — decides what to throw away from the cell state.
2. **Input gate** — decides what new information from the current input to add to the cell state.
3. **Output gate** — decides what part of the (updated) cell state to expose as this step's hidden state.

Because the cell state's path allows gradients to flow through largely unchanged when the forget gate says "keep this," LSTMs can learn dependencies across hundreds of steps that vanilla RNNs cannot.

In [3]:
import torch
import torch.nn as nn

lstm = nn.LSTM(input_size=5, hidden_size=8, num_layers=1, batch_first=True)

x = torch.randn(1, 10, 5)   # (batch, seq_len, input_size)
output, (h_n, c_n) = lstm(x)
print(output.shape)   # (1, 10, 8) -- hidden state at every time step
print(h_n.shape)       # (1, 1, 8)  -- final hidden state
print(c_n.shape)       # (1, 1, 8)  -- final cell state (the "long-term memory")


torch.Size([1, 10, 8])
torch.Size([1, 1, 8])
torch.Size([1, 1, 8])


## 24.5 — Building a Sequence Classifier With nn.LSTM

A common pattern: embed a sequence of token ids into vectors, run them through an LSTM, then feed only the **final hidden state** — a fixed-size summary of the whole sequence — into a `Linear` layer to produce a prediction. This is **sequence-to-one**: many inputs (one per time step), one output (for the whole sequence).

In [4]:
import torch
import torch.nn as nn

class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(input_size=embed_dim, hidden_size=hidden_size, num_layers=1, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)                  # (batch, seq_len) -> (batch, seq_len, embed_dim)
        output, (h_n, c_n) = self.lstm(embedded)       # h_n: (num_layers, batch, hidden_size)
        last_hidden = h_n[-1]                          # (batch, hidden_size) -- summary of the whole sequence
        return self.fc(last_hidden)                    # (batch, num_classes)

model = SentimentLSTM(vocab_size=5000, embed_dim=32, hidden_size=64, num_classes=2)
print(f'Total parameters: {sum(p.numel() for p in model.parameters())}')


Total parameters: 185218


## 24.6 — Sequence-to-One vs. Sequence-to-Sequence

Not every task wants just the final hidden state:

| Framing | Uses | Example task |
|---|---|---|
| Sequence → One | `h_n[-1]` (final hidden state only) | Sentiment classification, spam detection |
| Sequence → Sequence | `output` (hidden state at every step) | Part-of-speech tagging, next-word prediction |
| Sequence → Sequence (different length) | an encoder LSTM + a separate decoder LSTM | Machine translation, summarization |

> **Looking ahead: Chapter 25.** LSTMs process a sequence strictly one step at a time, which is slow to train and still struggles with very long-range dependencies. Transformers (next chapter) replace recurrence with **attention**, letting every position look directly at every other position.

## Chapter 24 Summary

1. Sequential data needs a **hidden state** carried across time steps — a memory of everything seen so far.
2. Vanilla RNNs suffer from the **vanishing gradient problem**, making it hard to learn dependencies more than ~10-20 steps back.
3. **LSTM**'s forget/input/output gates and separate cell state let gradients flow across hundreds of steps.
4. `nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)` returns `output, (h_n, c_n)`.
5. **Sequence-to-one** tasks use the final hidden state `h_n[-1]`; **sequence-to-sequence** tasks use every time step's output.

---

## Mini Project: LSTM Sentiment Classifier

The lesson's project trains `SentimentLSTM` on a small toy set of positive/negative sentences in Colab. To keep this notebook **fast, offline, and 100% reproducible**, we build an equivalent but fully synthetic sequence classification task with the exact same architecture and training loop shape:

**Synthetic task:** each sequence is 8 random "token ids" in `[0, 9]`. Label = 1 if the sequence's values sum to more than half of the maximum possible sum, else 0 — a task that genuinely requires the LSTM to accumulate information across the whole sequence (not just look at one token), generated entirely offline with `torch.randint`.

**What we build and actually run:**
- `SentimentLSTM(vocab_size=10, embed_dim=16, hidden_size=32, num_classes=2)` — the exact architecture from 24.5
- Confirm `model(torch.randint(0, vocab_size, (1, 8))).shape == torch.Size([1, 2])`
- `criterion = nn.CrossEntropyLoss()`, `optimizer = optim.Adam(model.parameters(), lr=0.005)`
- Train for several epochs and print loss + held-out test accuracy, confirming loss decreases and accuracy improves

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(0)

VOCAB_SIZE = 10   # token ids 0..9 directly represent values
SEQ_LEN = 8
THRESHOLD = SEQ_LEN * 4.5   # roughly half of the max possible sum (9*8=72) -> 36

def make_dataset(n):
    """Synthetic 'does this sequence sum to more than THRESHOLD' task."""
    seqs = torch.randint(0, VOCAB_SIZE, (n, SEQ_LEN))
    sums = seqs.float().sum(dim=1)
    labels = (sums > THRESHOLD).long()
    return seqs, labels

X_train, y_train = make_dataset(500)
X_test, y_test = make_dataset(120)
print('train label balance (fraction label=1):', y_train.float().mean().item())

clf = SentimentLSTM(vocab_size=VOCAB_SIZE, embed_dim=16, hidden_size=32, num_classes=2)
print(clf(torch.randint(0, VOCAB_SIZE, (1, SEQ_LEN))).shape)   # torch.Size([1, 2])

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(clf.parameters(), lr=0.005)

batch_size = 32
n_train = X_train.shape[0]

clf.eval()
with torch.no_grad():
    preds = torch.argmax(clf(X_test), dim=1)
    acc0 = (preds == y_test).float().mean().item() * 100
print(f'Before training, test accuracy: {acc0:.2f}%')

for epoch in range(15):
    clf.train()
    perm = torch.randperm(n_train)
    running_loss = 0.0
    n_batches = 0
    for i in range(0, n_train, batch_size):
        idx = perm[i:i+batch_size]
        seqs, labels = X_train[idx], y_train[idx]
        optimizer.zero_grad()
        outputs = clf(seqs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        n_batches += 1
    clf.eval()
    with torch.no_grad():
        preds = torch.argmax(clf(X_test), dim=1)
        acc = (preds == y_test).float().mean().item() * 100
    if epoch % 2 == 0 or epoch == 14:
        print(f'Epoch {epoch+1}, avg loss {running_loss/n_batches:.4f}, test accuracy {acc:.2f}%')


train label balance (fraction label=1): 0.49399998784065247
torch.Size([1, 2])


Before training, test accuracy: 55.00%
Epoch 1, avg loss 0.6639, test accuracy 70.83%
Epoch 3, avg loss 0.2190, test accuracy 93.33%
Epoch 5, avg loss 0.0729, test accuracy 94.17%


Epoch 7, avg loss 0.0696, test accuracy 95.83%
Epoch 9, avg loss 0.0478, test accuracy 95.83%
Epoch 11, avg loss 0.0227, test accuracy 95.00%
Epoch 13, avg loss 0.0088, test accuracy 95.00%


Epoch 15, avg loss 0.0059, test accuracy 93.33%


Loss drops from ~0.64 to well under 0.05 and test accuracy climbs from near-random (~50%) to the mid-90s% — confirming the LSTM is genuinely accumulating information across all 8 time steps to make its prediction, not just memorizing one token.

## Next: Chapter 25 — Transformers (runs in this browser!)